# V58 - Dense embedding screening

Pre-registered plan: sparse TF-IDF plateaus near 0.627, so test dense multilingual-E5 representations of the same chats, the same syllabus-cosine idea in dense space, and their stacks with the word-only baseline. E5 needs its query prefix and internet on first run to fetch weights. One adoption at most, gated as usual. Empty-chat users embed the empty string, a known weak point recorded here rather than hidden.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics.pairwise import cosine_similarity

SEED = 808
N_SPLITS = 5
GATE_FLOOR = 0.002
MODEL = "intfloat/multilingual-e5-base"
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v58")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Embed chats and syllabi

One encoder, E5 query prefix on user chats, plain syllabus texts. Embeddings are cached to disk so reruns skip the encoder.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
syll = [catalog.set_index("module_id").loc[m, "description_and_syllabus"] if m in set(catalog["module_id"]) else "" for m in MODS]

cache_u, cache_s = OUT / "e5_users.npy", OUT / "e5_syll.npy"
if cache_u.exists() and cache_s.exists():
    E, ES = np.load(cache_u), np.load(cache_s)
    print("loaded cached embeddings", E.shape)
else:
    enc = SentenceTransformer(MODEL)
    E = enc.encode(["query: " + d for d in docs], batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    ES = enc.encode(syll, batch_size=8, show_progress_bar=True, normalize_embeddings=True)
    np.save(cache_u, E)
    np.save(cache_s, ES)
    print("encoded", E.shape)
E_tr, E_te = E[:n_train], E[n_train:]
CE = E_tr @ ES.T
CE_te = E_te @ ES.T

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
A_tr = A[:n_train]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
W_tr = W[:n_train]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

loaded cached embeddings (5000, 768)


## Gate dense candidates

Same locks as ever against the word-only baseline.

In [3]:
def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(Xtr, alpha=1.0):
    oof = np.zeros_like(Y)
    fids = np.zeros(len(Xtr), int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha, random_state=SEED)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
        fids[va] = fi + 1
    return oof, fids

oof_w, fid = ridge_oof(W_tr)
base = ndcg5(Y, oof_w)
print(f"baseline word-only OOF: {base.mean():.5f}")
cands = {"E5 dense": E_tr, "E5 + assessment": np.hstack([E_tr, A_tr]),
         "E5 + word-only": np.hstack([E_tr, W_tr]), "content-E5 cosine": CE}
best = ("baseline", 0.0)
for name, Xtr in cands.items():
    oo, _ = ridge_oof(Xtr)
    d = ndcg5(Y, oo) - base
    lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
    wf = min(ndcg5(Y[fid == k], oo[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
    ok = (lo > GATE_FLOOR) and (wf >= 0)
    print(f"{name}: OOF={ndcg5(Y, oo).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f} {'ADOPT' if ok else 'reject'}")
    if ok and d.mean() > best[1]:
        best = (name, d.mean())
print("adopted:", best[0])

baseline word-only OOF: 0.62606


E5 dense: OOF=0.61477 delta=-0.01128 CI95=[-0.01497,-0.00754] worst-fold=-0.01850 reject


E5 + assessment: OOF=0.61677 delta=-0.00929 CI95=[-0.01306,-0.00550] worst-fold=-0.01763 reject


E5 + word-only: OOF=0.62548 delta=-0.00058 CI95=[-0.00260,+0.00148] worst-fold=-0.00354 reject


content-E5 cosine: OOF=0.48212 delta=-0.14394 CI95=[-0.15065,-0.13724] worst-fold=-0.18610 reject
adopted: baseline
